# Retrain the prop model on Colab

Runs the real trainer — `backend/prop_modeling/train_regression_model.py` — on a Colab
runtime, so an hour-long fit does not tie up a laptop.

**Pick a CPU runtime, not a GPU one.** CatBoost's `MultiQuantile` loss is CPU-only; the
trainer pins `task_type="CPU"` for that reason. A GPU runtime costs quota and changes
nothing. What matters is vCPU count and RAM.

**Why a bundle instead of `git clone`.** `backend/data` is gitignored, so a clone brings
the code and none of the inputs. `scripts/make_colab_bundle.py` packs both — ~4 MB gzipped,
because it ships the 17 MB of *inputs* rather than the 287 MB derived dataset and rebuilds
the big file here.

**The catboost version is a correctness requirement, not a preference.**
`requirements-vm.txt` pins `catboost==1.2.10`, and the VM has to load the `.cbm` files this
notebook produces. The install cell pins the same version.

Run the cells in order. Each one prints enough to tell whether to continue.

## 1. Runtime check

Prints what you actually got, and says so if a GPU runtime was selected — which for this
loss function is strictly worse than CPU.

In [ ]:
import os, platform, shutil, subprocess, sys

print("python     ", sys.version.split()[0])
print("platform   ", platform.platform())
try:
    import multiprocessing; print("cpu count  ", multiprocessing.cpu_count())
except Exception:
    pass
try:
    with open("/proc/meminfo") as fh:
        total = next(l for l in fh if l.startswith("MemTotal"))
    print("memory     ", total.split()[1:])
except Exception:
    pass

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
print("in colab   ", IN_COLAB)

if shutil.which("nvidia-smi"):
    print("\nA GPU runtime is attached. CatBoost MultiQuantile is CPU-only, so the GPU")
    print("will sit idle and GPU runtimes often have FEWER vCPUs. Switch to CPU:")
    print("  Runtime > Change runtime type > Hardware accelerator: None")
else:
    print("\nCPU runtime - correct for this job.")

## 2. Pin dependencies

Colab ships its own pandas/numpy, and this repo is on pandas 3. Pinning avoids a silent
behaviour difference between what you test locally and what trains here.

If pip reports a version conflict it wants a restart for, restart the runtime and re-run
from this cell — the later cells are all idempotent.

In [ ]:
import subprocess, sys

PINS = [
    "catboost==1.2.10",   # must match requirements-vm.txt: the VM loads these .cbm files
    "pandas==3.0.0",
    "numpy==2.4.2",
    "scikit-learn==1.8.0",
    "python-dotenv==1.2.1",
]
print("installing:", " ".join(PINS))
out = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS],
                     capture_output=True, text=True)
print(out.stdout[-2000:] or "(quiet)")
if out.returncode != 0:
    print("FAILED:\n", out.stderr[-3000:])
else:
    import importlib.metadata as md
    for p in ("catboost", "pandas", "numpy", "scikit-learn"):
        print("  %-14s %s" % (p, md.version(p)))
    print("\nIf pandas/numpy changed, restart the runtime now, then re-run from here.")

## 3. Get the bundle onto the runtime

Build it locally first:

```bash
cd backend
python3 scripts/sync_all_data.py          # make sure the inputs are current
python3 scripts/make_colab_bundle.py      # -> backend/generated/colab_input.tar.gz
```

Then pick **one** of these:

**A — Upload from VS Code (no Drive, no auth).** In the Explorer, right-click
`backend/generated/colab_input.tar.gz` → **Upload to Colab**. It lands in `/content/`.
Leave `BUNDLE` as the default below.

**B — Google Drive.** Run `Colab: Mount Google Drive to Server...` from the command
palette, or let the cell mount it. Put the tarball anywhere in Drive and set `BUNDLE` to
its path. Use this if you expect to retrain repeatedly — the upload survives a runtime
reset, where option A does not.

The cell searches the usual places, so in most cases you can just run it.

In [ ]:
import hashlib, os, sys, tarfile
from pathlib import Path

# Set explicitly to override the search below.
BUNDLE = None
WORKDIR = Path("/content/nba")

CANDIDATES = [
    Path("/content/colab_input.tar.gz"),              # "Upload to Colab" lands here
    Path("/content/drive/MyDrive/colab_input.tar.gz"),
    Path("colab_input.tar.gz"),
    Path("backend/generated/colab_input.tar.gz"),     # running locally, not on Colab
]

bundle = Path(BUNDLE) if BUNDLE else next((p for p in CANDIDATES if p.exists()), None)

# Only mount Drive if the bundle is not already here -- mounting prompts for
# authorisation, and option A does not need it.
if bundle is None and "google.colab" in sys.modules:
    print("Bundle not found locally; mounting Drive to look there...")
    from google.colab import drive
    drive.mount("/content/drive")
    bundle = next((p for p in CANDIDATES if p.exists()), None)
    if bundle is None:
        found = sorted(Path("/content/drive/MyDrive").rglob("colab_input.tar.gz"))[:5]
        if found:
            bundle = found[0]
            print("found %s" % bundle)

if bundle is None:
    raise SystemExit(
        "No bundle found. Either right-click backend/generated/colab_input.tar.gz in the "
        "VS Code Explorer and choose 'Upload to Colab', or put it in Drive and set BUNDLE "
        "to its path.\nLooked in:\n  " + "\n  ".join(str(p) for p in CANDIDATES)
    )

digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
print("bundle      %s (%.1f MB)" % (bundle, bundle.stat().st_size / 1e6))
print("sha256[:16] %s" % digest[:16])

WORKDIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(bundle) as tar:
    tar.extractall(WORKDIR)
BACKEND = WORKDIR / "backend"
if not BACKEND.is_dir():
    raise SystemExit("Unpacked %s but found no backend/ inside it." % bundle)
print("unpacked to %s" % BACKEND)

# Carried into the exported metadata: git_sha is null here because the runtime
# has an archive rather than a checkout, so this is the only provenance link
# back to the code and data that produced the weights.
BUNDLE_SHA = digest

## 4. Check the inputs before spending an hour

The single most common cause of a surprising result is training on stale game logs. This
prints how far behind today each one is.

In [ ]:
import csv, sys
from datetime import date
from pathlib import Path

sys.path[:0] = [str(BACKEND), str(BACKEND / "prop_modeling")]
from scrapers.season_type import season_type_from_game_id

for path in sorted((BACKEND / "data" / "archive").glob("gamelogs_*.csv")) + \
            sorted((BACKEND / "data" / "current").glob("gamelogs_*.csv")):
    if ".backup." in path.name or "failed" in path.name:
        continue
    rows = 0; lo = hi = None; kinds = {}
    with path.open(newline="") as fh:
        for row in csv.DictReader(fh):
            rows += 1
            d = (row.get("GAME_DATE") or "")[:10]
            if d:
                lo = d if lo is None or d < lo else lo
                hi = d if hi is None or d > hi else hi
            k = season_type_from_game_id(row.get("GAME_ID"))
            kinds[k] = kinds.get(k, 0) + 1
    lag = (date.today() - date.fromisoformat(hi)).days if hi else None
    print("%-26s %6d rows  %s .. %s  (%s days behind)  %s"
          % (path.name, rows, lo, hi, lag, kinds))

print("\nPlayoff rows are dropped by the builder (regular season only).")
print("If the current-season log is more than a day or two behind, re-run")
print("scripts/sync_all_data.py locally and rebuild the bundle before continuing.")

## 5. Rebuild the dataset

Regenerates `regression_training_dataset.csv` and `minutes_training_dataset.csv` from the
game logs. A few minutes. Drops playoff/play-in/all-star rows and prints what it dropped.

In [ ]:
import subprocess, sys, time

t0 = time.time()
proc = subprocess.run(
    [sys.executable, "-u", "prop_modeling/build_regression_dataset.py"],
    cwd=str(BACKEND), capture_output=True, text=True,
)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print("FAILED\n", proc.stderr[-4000:])
    raise SystemExit(proc.returncode)
print("\nbuild took %.1f min" % ((time.time() - t0) / 60))

## 6. Train

Defaults are deliberate:

- **no `--tune`.** The last measured Optuna pass bought 0.9% of PTS MAE for 40 trials — and
  it was selecting on the test set at the time. Add `--tune --tune-trials 30` if you want it.
- **`--allow-feature-change`** is required: the current dataset yields 123 features against
  the shipped bundle's 117. All six additions are verified producible at serve time, which is
  why passing it is safe *here* and should not be habitual.
- **walk-forward current-season evaluation is on.** Early in a season the trailing holdout
  cannot reach the current season at all, so this is the only in-regime number.

Expect roughly an hour for all eleven heads; the minutes model is fitted once and shared.

If you would rather watch it stream than wait on a cell, run `Colab: Open Terminal` from
the command palette and run the same command there:

```bash
cd /content/nba/backend && python3 -u prop_modeling/train_regression_model.py \
    --model-dir prop_modeling/exported_regression_model --allow-feature-change
```

A terminal also survives the notebook losing focus, which on a long run matters.


In [ ]:
import subprocess, sys, time

ARGS = [
    "--model-dir", "prop_modeling/exported_regression_model",
    "--allow-feature-change",
    # "--tune", "--tune-trials", "30",
    # "--stats", "PTS", "REB",     # uncomment for a fast smoke run
]

t0 = time.time()
proc = subprocess.run(
    [sys.executable, "-u", "prop_modeling/train_regression_model.py", *ARGS],
    cwd=str(BACKEND), capture_output=True, text=True,
)
print(proc.stdout[-20000:])
if proc.returncode != 0:
    print("FAILED\n", proc.stderr[-4000:])
    raise SystemExit(proc.returncode)
print("\ntraining took %.1f min" % ((time.time() - t0) / 60))

## 7. Verify the bundle before trusting it

Two checks that have both caught real problems:

1. every `.cbm` loads, and its own `feature_names_` match `model_metadata.json` positionally
   — a mismatch predicts from a shifted vector and returns plausible wrong numbers with no
   exception;
2. every head beats the player's own recent-10 average. One that does not is not adding
   information.

In [ ]:
import json
from pathlib import Path
from catboost import CatBoostRegressor

MODEL_DIR = BACKEND / "prop_modeling" / "exported_regression_model"
meta = json.loads((MODEL_DIR / "model_metadata.json").read_text())
declared = list(meta["feature_columns"])
print("model_type %s | %d features | alphas %s"
      % (meta["model_type"], len(declared), meta["quantile_alphas"]))
print("shipped_fit %s" % meta.get("provenance", {}).get("shipped_fit"))
print("")

bad = []
for stat, fname in sorted(meta["model_files"].items()):
    m = CatBoostRegressor(); m.load_model(str(MODEL_DIR / fname))
    names = list(getattr(m, "feature_names_", None) or [])
    ok = (not names) or names == declared
    if not ok:
        bad.append(stat)
    print("  %-13s %-28s trees=%5d  feature_names %s"
          % (stat, fname, m.tree_count_ or 0, "OK" if ok else "MISMATCH"))
if bad:
    print("\nMISMATCHED: %s -- do not ship this bundle." % ", ".join(bad))

print("\n%-13s %7s %9s %9s %9s %9s" % ("stat", "n", "MAE q50", "bias", "vs rec10", "cur-szn"))
losers = []
for stat in sorted(meta["per_stat_test_metrics"]):
    t = meta["per_stat_test_metrics"][stat]
    d = ((t.get("baselines") or {}).get("recent10") or {}).get("delta")
    cs = (meta.get("per_stat_current_season_metrics") or {}).get(stat) or {}
    if d is not None and d > 0:
        losers.append(stat)
    print("  %-13s %7d %9.4f %+9.4f %9s %9s"
          % (stat, t.get("n") or 0, t.get("mae_q50") or float("nan"),
             t.get("q50_bias") or float("nan"),
             ("%+.4f" % d) if d is not None else "n/a",
             ("%.3f" % cs["mae_q50"]) if cs.get("mae_q50") is not None else "n/a"))
if losers:
    print("\nLoses to the recent-10 baseline on: %s" % ", ".join(losers))

## 8. Write the bundle back to Drive

Stamps the bundle digest into the metadata first — `git_sha` is null on Colab because the
input is an archive, not a checkout, so this is the only link back to what was trained.

In [ ]:
import json, shutil, tarfile
from datetime import datetime, timezone
from pathlib import Path

meta_path = MODEL_DIR / "model_metadata.json"
meta = json.loads(meta_path.read_text())
meta.setdefault("provenance", {})["colab"] = {
    "input_bundle_sha256": BUNDLE_SHA,
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "note": "Trained on Colab from an input bundle; git_sha is null because the "
            "runtime had no checkout. input_bundle_sha256 identifies the code and "
            "data that produced these weights.",
}
meta_path.write_text(json.dumps(meta, indent=2, sort_keys=True))

OUT = Path("/content/drive/MyDrive/exported_regression_model.tar.gz")
with tarfile.open(OUT, "w:gz") as tar:
    tar.add(MODEL_DIR, arcname="exported_regression_model")
print("wrote %s (%.1f MB)" % (OUT, OUT.stat().st_size / 1e6))
print("bundle sha256[:16] = %s" % BUNDLE_SHA[:16])

## 9. Back on the laptop — gate it before committing

```bash
cd backend
tar -xzf ~/Downloads/exported_regression_model.tar.gz -C prop_modeling/   # overwrites the bundle
python3 -m pytest tests/ -q
python3 prop_modeling/backtest.py --source supabase --model
```

The gate is the decision, not the training log. What to look for:

- **`mae_delta` against the closing line.** Positive means the line is the better
  projection. The bundle this replaces lost on every PTS market by +0.26 to +0.46.
- **`q50-act` bias.** The closing line's own is −0.08 to +0.02; the old bundle's was −1.53.
- **log-loss vs the de-vigged market**, reported but thin — gate on calibration, report ROI.

Keep the previous `exported_regression_model/` until the gate passes. A push to `main` is a
production deploy of both Vercel and the pipeline, so commit the `.cbm` files and
`model_metadata.json` in the same commit — they are only meaningful together.